In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth, fpmax

In [2]:
data = pd.read_csv('dataset.csv')
data = data.sort_values(['timestamp'])

In [3]:
train = data[:80000]
test = data[80000:]

In [4]:
train.head()

,user_id,item_id,rating,timestamp
217,259,255,4,874724710
83968,259,286,4,874724727
43030,259,298,4,874724754
21399,259,185,4,874724781
82658,259,173,4,874724843


In [5]:
test.head()

,user_id,item_id,rating,timestamp
1346,3,245,1,889237247
27978,3,355,3,889237247
1260,3,335,1,889237269
38673,3,322,3,889237269
3761,3,323,2,889237269


In [6]:
def average_precision(actual, recommended, k=30):
    ap_sum = 0
    hits = 0
    for i in range(k):
        product_id = recommended[i] if i < len(recommended) else None
        if product_id is not None and product_id in actual:
            hits += 1
            ap_sum += hits / (i + 1)
    return ap_sum / k


def normalized_average_precision(actual, recommended, k=30):
    actual = set(actual)
    if len(actual) == 0:
        return 0.0

    ap = average_precision(actual, recommended, k=k)
    ap_ideal = average_precision(actual, list(actual)[:k], k=k)
    return ap / ap_ideal

In [7]:
def recommend(user):
    return [288, 1, 286, 121, 174]

In [8]:
scores = []
for user in tqdm(test['user_id'].unique()):
    actual = list(test[test['user_id'] == user]['item_id'])
    recommended = recommend(user)

    scores.append(normalized_average_precision(actual, recommended))

np.mean(scores)

100%|██████████| 301/301 [00:00<00:00, 594.30it/s]


0.03566965142495101

In [9]:
# Задача: Обучить модель так, чтобы мера была больше 0.1

In [10]:
# Уберем курсы с низким рейтингом
my_train = train[train['rating'] > 2]
my_train = my_train.astype({'item_id': np.str})

data_by_user = my_train.groupby(['user_id'])['item_id'].apply(lambda x: ','.join(x).strip()).reset_index()
data_by_user 

,user_id,item_id
0,0,"172,50"
1,1,"168,172,165,156,166,196,187,127,14,250,181,109..."
2,2,"286,258,305,307,288,312,301,306,269,303,292,29..."
3,3,"344,268,303,345,354,350,334,343,339,342,299,30..."
4,5,"267,455,222,121,363,405,257,250,25,21,100,109,..."
...,...,...
747,937,"242,286,258,303,300,304,874,100,14,116,283,124..."
748,939,"326,689,258,127,9,257,275,993,1190,742,222,591..."
749,940,"286,310,301,683,347,315,289,751,259,321,300,26..."
750,941,"300,258,294,117,408,298,919,181,257,7,993,763,..."


In [11]:
# Данные готовы для поиска ассоциативных правил
dummy_data_by_user = data_by_user['item_id'].str.get_dummies(',')
dummy_data_by_user

,1,10,100,1000,1001,1002,1003,1004,1005,1006,...,990,991,992,993,994,995,996,997,998,999
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
748,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
749,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
750,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [12]:
# Используем алгоритм априори
frequent_itemsets_user = apriori(dummy_data_by_user, min_support=0.15, use_colnames=True)

frequent_itemsets_user

,support,itemsets
0,0.461436,(1)
1,0.519947,(100)
2,0.231383,(11)
3,0.276596,(111)
4,0.372340,(117)
...,...,...
8945,0.150266,"(195, 210, 50, 79, 174, 172, 181)"
8946,0.150266,"(69, 204, 210, 50, 174, 172, 181)"
8947,0.160904,"(204, 210, 50, 79, 174, 172, 181)"
8948,0.155585,"(98, 204, 210, 50, 174, 172, 181)"


In [13]:
# Получили ассоциативные правила с поддержкой 0.15 и уверенностью 0.6
# Это краткий обзор, как будет работать модель
rules_by_user = association_rules(frequent_itemsets_user, metric="confidence", min_threshold=0.6)
antecedents = list(rules_by_user['antecedents'])
consequents = list(rules_by_user['consequents'])
rules_by_user

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
0,(1),(100),0.461436,0.519947,0.303191,0.657061,1.263707,0.063269,1.399819
1,(111),(1),0.276596,0.461436,0.199468,0.721154,1.562846,0.071837,1.931401
2,(117),(1),0.372340,0.461436,0.265957,0.714286,1.547962,0.094146,1.884973
3,(118),(1),0.251330,0.461436,0.198138,0.788360,1.708492,0.082166,2.544714
4,(1),(121),0.461436,0.381649,0.280585,0.608069,1.593268,0.104478,1.577705
...,...,...,...,...,...,...,...,...,...
64269,"(79, 50, 172)","(204, 174, 98, 181)",0.248670,0.203457,0.151596,0.609626,2.996330,0.101002,2.040458
64270,"(79, 174, 172)","(204, 98, 50, 181)",0.243351,0.211436,0.151596,0.622951,2.946283,0.100143,2.091408
64271,"(79, 174, 181)","(204, 98, 50, 172)",0.251330,0.211436,0.151596,0.603175,2.852750,0.098456,1.987181
64272,"(79, 172, 181)","(204, 174, 98, 50)",0.235372,0.219415,0.151596,0.644068,2.935388,0.099952,2.193072


In [14]:
# Функция подбирает рекомендации с помощью полученных правил
# Для новых пользователей предлагаем рекомендованный набор с полученной экспериментально минимальной поддержкой
# Пользователям, которым не смогли порекомендовать что-то, предлагаем курсы из рекомендованного набора
def recommend2(user):
    recommended_set = set()
    user_set = frozenset(my_train[my_train['user_id'] == user]['item_id'])
    for ind in range(len(antecedents)):
        if antecedents[ind].issubset(user_set):
            for item in consequents[ind]:
                if item not in user_set:
                    recommended_set.add(int(item))
    if len(user_set) == 0:
        return recommended_list_for_new_user
    if len(recommended_set) == 0:
        for item in recommended_list_for_new_user:
            if item not in user_set:
                recommended_set.add(int(item))
    #print(user, sorted(list(recommended_set)), actual)
    return list(recommended_set)

In [15]:
recommended_list_for_new_user = list()
antecedents = list()
consequents = list()
max_score = 0
p_min_support = 0
p_min_confidence = 0

# Значение получено перебором
best_val_for_standart_list = 0.31
frequent_itemsets_user_for_standart = apriori(dummy_data_by_user, min_support = best_val_for_standart_list, use_colnames = True)
itemsets_for_standart = list(frequent_itemsets_user_for_standart['itemsets'])
recommended_set_for_new_user = set()
for itemset in itemsets_for_standart:
    for item in itemset:
        recommended_set_for_new_user.add(int(item))
recommended_list_for_new_user = list(recommended_set_for_new_user)

# Подбор параметров
for min_support_val in tqdm(np.arange(0.15,0.35,0.01)):
    frequent_itemsets_user = apriori(dummy_data_by_user, min_support = min_support_val, use_colnames = True)
    for min_confidence_val in np.arange(0.6,0.95,0.01):
        rules_by_user = association_rules(frequent_itemsets_user, metric = 'confidence', min_threshold = min_confidence_val)
        antecedents = list(rules_by_user['antecedents'])
        consequents = list(rules_by_user['consequents'])
        
        scores = []
        for user in test['user_id'].unique():
            actual = list(test[test['user_id'] == user]['item_id'])
            recommended = recommend2(user)

            scores.append(normalized_average_precision(actual, recommended))

        total = np.mean(scores)
        if total > max_score:
            max_score = total
            p_min_support = min_support_val
            p_min_confidence = min_confidence_val
        #print(min_support_val, min_confidence_val, total)

            
            
print(p_min_support, p_min_confidence, max_score)

100%|██████████| 20/20 [09:53<00:00, 29.67s/it]

0.15 0.6 0.1443349205791958


In [16]:
# Используем полученные значения
print('min_support =', p_min_support, 'min_confidence =', p_min_confidence)
frequent_itemsets_user = apriori(dummy_data_by_user, min_support = p_min_support, use_colnames = True)
rules_by_user = association_rules(frequent_itemsets_user, metric = 'confidence', min_threshold = p_min_confidence)
antecedents = list(rules_by_user['antecedents'])
consequents = list(rules_by_user['consequents'])

min_support = 0.15 min_confidence = 0.6


In [17]:
# Итоговый результат
scores = []
for user in tqdm(test['user_id'].unique()):
    actual = list(test[test['user_id'] == user]['item_id'])
    recommended = recommend2(user)

    scores.append(normalized_average_precision(actual, recommended))

np.mean(scores)

100%|██████████| 301/301 [00:05<00:00, 59.05it/s]


0.1443349205791958